# Inspect the header of the dataset file

In [9]:
!head -n 5 complex.tsv

#Complex ac	Recommended name	Aliases for complex	Taxonomy identifier	Identifiers (and stoichiometry) of molecules in complex	Evidence Code	Experimental evidence	Go Annotations	Cross references	Description	Complex properties	Complex assembly	Ligand	Disease	Agonist	Antagonist	Comment	Source	Expanded participant list
CPX-1	SMAD2-SMAD3-SMAD4 complex	SMAD2/SMAD3/SMAD4 transcription factor complex	9606	P84022(1)|Q13485(1)|Q15796(1)	ECO:0005547(biological system reconstruction evidence based on inference from background scientific knowledge used in manual assertion)	-	GO:0071144(heteromeric SMAD protein complex)|GO:0003690(double-stranded DNA binding)|GO:0003700(DNA-binding transcription factor activity)|GO:0006355(regulation of DNA-templated transcription)|GO:0032924(activin receptor signaling pathway)|GO:0007179(transforming growth factor beta receptor signaling pathway)	reactome:R-HSA-9736938(identity)|reactome:R-HSA-9736929(identity)|pubmed:35359452(see-also)|pubmed:16322555(see-also)|com

# Install NetworkX Library

In [6]:
!pip install networkx

# Generate Graph and Hypergraph from IntAct Data

In [10]:
import pandas as pd
import networkx as nx
import re
from itertools import combinations

def create_network_representations(intact_file_path):
    """
    Parses the IntAct complex.tsv file to create both a graph and a hypergraph.
    This version explicitly defines column names to prevent parsing errors.
    """
    try:
        # --- 1. Define the correct column names based on the file specification ---
        # This is the crucial fix. We provide the header manually.
        column_names = [
            '#Complex ac',
            'Recommended name',
            'Aliases for complex',
            'Taxonomy identifier',
            'Identifiers (and stoichiometry) of molecules in complex',
            'Evidence Code',
            'Experimental evidence',
            'Go Annotations',
            'Cross references',
            'Description',
            'Complex properties',
            'Complex assembly',
            'Ligand',
            'Disease',
            'Agonist',
            'Antagonist',
            'Comment',
            'Source',
            'Expanded participant list'
        ]

        # --- 2. Load Data with Explicit Instructions ---
        #  - header=0: The first line of the file is the header.
        #  - names=column_names: But we override it with our list.
        #  - comment='#': Still useful for other commented lines, though the header is the main one.
        #  - sep='\t': Specify the delimiter.
        # We read the file by telling it the first line is a header to be skipped,
        # and we provide the correct names.
        df = pd.read_csv(
            intact_file_path,
            sep='\t',
            header=0, # The file has a header on the first line
            names=column_names, # We rename the columns with our list
            comment='#' # This handles the '#' at the start of the header line
        )

        # --- 3. Filter and Process the Data ---
        df_human = df[df['Taxonomy identifier'] == 9606].copy()
        df_human.dropna(subset=['Identifiers (and stoichiometry) of molecules in complex'], inplace=True)
        print(f"Loaded and filtered to {len(df_human)} human complexes.")

        # --- 4. Build Hypergraph Representation ---
        H = {}
        uniprot_pattern = re.compile(r"([A-Z0-9]{6,})")

        for index, row in df_human.iterrows():
            complex_id = row['#Complex ac'] # Use the name from our list
            identifiers_str = row['Identifiers (and stoichiometry) of molecules in complex']

            protein_ids = set(uniprot_pattern.findall(identifiers_str))

            if len(protein_ids) >= 2:
                H[complex_id] = protein_ids

        print(f"Created a hypergraph with {len(H)} hyperedges (complexes).")

        # --- 5. Build Graph Representation (Pairwise Projection) ---
        G = nx.Graph()
        for complex_id, proteins_in_complex in H.items():
            edges = combinations(proteins_in_complex, 2)
            G.add_edges_from(edges)

        print(f"Created a graph with {G.number_of_nodes()} nodes (proteins) and {G.number_of_edges()} edges.")

        return G, H

    except FileNotFoundError:
        print(f"ERROR: The file was not found at {intact_file_path}. Please upload it.")
        return None, None
    except KeyError as e:
        print(f"ERROR: A required column was not found: {e}.")
        if 'df' in locals():
            print("Available columns:", df.columns.tolist())
        return None, None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None

# --- USAGE ---
intact_file = "complex.tsv"
graph, hypergraph = create_network_representations(intact_file)

# --- Inspect the Results ---
if graph and hypergraph:
    print("\n--- Inspection ---")
    print(f"Graph representation (G) is a NetworkX object: {type(graph)}")
    print(f"Hypergraph representation (H) is a dict: {type(hypergraph)}")

    # Example check
    example_protein_id = 'P84022' # SMAD4
    if example_protein_id in graph:
        neighbors = list(graph.neighbors(example_protein_id))
        print(f"Neighbors of protein '{example_protein_id}' in the graph: {neighbors[:5]}...")

Loaded and filtered to 2345 human complexes.
Created a hypergraph with 2135 hyperedges (complexes).
Created a graph with 3580 nodes (proteins) and 19159 edges.

--- Inspection ---
Graph representation (G) is a NetworkX object: <class 'networkx.classes.graph.Graph'>
Hypergraph representation (H) is a dict: <class 'dict'>
Neighbors of protein 'P84022' in the graph: ['P43699', 'Q9NZH6', 'Q13485']...


# Save Network Representations to Files

In [11]:
graph_output_path = "intact_network.graph"
hypergraph_output_path = "intact_network.hypergraph"

try:
    # --- 1. Save the Graph as an Edgelist ---
    # The edgelist format is a simple text file where each line is "node1 node2".
    # It's a standard way to represent simple graphs.
    nx.write_edgelist(graph, graph_output_path, data=False)
    print(f"Graph saved successfully as an edgelist to: {graph_output_path}")


    # --- 2. Save the Hypergraph in a Custom Text Format ---
    # We will write one line per hyperedge (complex).
    # Format: ComplexID Member1 Member2 Member3 ...
    with open(hypergraph_output_path, 'w') as f:
        # We iterate through each complex (key) and its set of proteins (value)
        for complex_id, proteins in hypergraph.items():
            # Sorting the proteins makes the output file consistent and easier to read
            protein_list = sorted(list(proteins))
            # Join all proteins with a space
            protein_str = " ".join(protein_list)
            # Write the line to the file
            f.write(f"{complex_id} {protein_str}\n")
    print(f"Hypergraph saved successfully in a custom format to: {hypergraph_output_path}")


    # --- 3. (Optional) Show a preview of the saved files ---
    print("\n--- Preview of .graph file (first 5 lines) ---")
    !head -n 5 {graph_output_path}

    print("\n--- Preview of .hypergraph file (first 5 lines) ---")
    !head -n 5 {hypergraph_output_path}

except NameError:
    print("ERROR: The 'graph' or 'hypergraph' objects were not found.")
    print("Please make sure you have run the previous cell to generate them first.")
except Exception as e:
    print(f"An unexpected error occurred during file saving: {e}")

Graph saved successfully as an edgelist to: intact_network.graph
Hypergraph saved successfully in a custom format to: intact_network.hypergraph

--- Preview of .graph file (first 5 lines) ---
P18848 P16220
P18848 P18846
P18848 P18847
P18848 P15336
P18848 Q16520

--- Preview of .hypergraph file (first 5 lines) ---
CPX-8 P16220 P18848
CPX-9 P18846 P18848
CPX-17 O14746 URS00004A7003
CPX-20 O14746 URS000075C8FA
CPX-37 P05109 P06702


# Prepare Network Files for Centrality Calculation

In [13]:
import networkx as nx

# --- Step 0: Ensure 'graph' and 'hypergraph' objects exist ---
try:
    if not isinstance(graph, nx.Graph) or not isinstance(hypergraph, dict):
        raise NameError("Graph/Hypergraph not found")
except NameError:
    print("ERROR: The 'graph' or 'hypergraph' objects were not found.")
    print("Please make sure you have run the cell that generates them first.")
    # Stop execution if objects don't exist
    assert False, "Prerequisite objects not found."


# --- Step 1: Create a Universal ID Mapping ---
# We need to map every unique string identifier (proteins, RNAs, etc.)
# to a unique integer so the C++ program can read them.

print("--- Step 1: Creating Universal ID Mapping ---")
all_unique_ids = set(graph.nodes()) # Start with all nodes from the graph
for complex_id, members in hypergraph.items():
    # Add the complex ID itself (though not used for C++ input, it's good practice)
    all_unique_ids.add(complex_id)
    # Add all members of the complex
    all_unique_ids.update(members)

# Create the mappings: string -> integer and integer -> string
string_to_int = {string_id: i for i, string_id in enumerate(all_unique_ids)}
int_to_string = {i: string_id for string_id, i in string_to_int.items()}

# Save the mapping for later. This is CRITICAL for interpreting the results.
mapping_output_path = "id_mapping.tsv"
with open(mapping_output_path, 'w') as f:
    f.write("IntegerID\tOriginalID\n")
    for int_id, string_id in int_to_string.items():
        f.write(f"{int_id}\t{string_id}\n")

print(f"Found {len(all_unique_ids)} unique identifiers.")
print(f"Mapping saved to: {mapping_output_path}\n")


# --- Step 2: Generate the .graph file for C++ ---
print("--- Step 2: Generating .graph file for C++ ---")
cpp_graph_path = "EBI-IntAct.graph"
with open(cpp_graph_path, 'w') as f:
    # Write the header line required by the C++ code
    num_nodes = graph.number_of_nodes()
    num_edges = graph.number_of_edges()
    f.write(f"{num_nodes} {num_edges}\n")

    # Write each edge using the new integer IDs
    for u, v in graph.edges():
        int_u = string_to_int[u]
        int_v = string_to_int[v]
        f.write(f"{int_u} {int_v}\n")

print(f"Graph file for C++ saved to: {cpp_graph_path}")
print("--- Preview of cpp_input.graph (first 5 lines) ---")
!head -n 5 {cpp_graph_path}
print("")


# --- Step 3: Generate the .hypergraph file for C++ ---
print("--- Step 3: Generating .hypergraph file for C++ ---")
cpp_hypergraph_path = "EBI_IntAct.hypergraph"
with open(cpp_hypergraph_path, 'w') as f:
    # For each complex, write a line containing the integer IDs of its members
    for complex_id, members in hypergraph.items():
        # Get integer IDs for each member
        int_members = [str(string_to_int[m]) for m in members]
        f.write(" ".join(int_members) + "\n")

print(f"Hypergraph file for C++ saved to: {cpp_hypergraph_path}")
print("--- Preview of cpp_input.hypergraph (first 5 lines) ---")
!head -n 5 {cpp_hypergraph_path}
print("")

--- Step 1: Creating Universal ID Mapping ---
Found 5715 unique identifiers.
Mapping saved to: id_mapping.tsv

--- Step 2: Generating .graph file for C++ ---
Graph file for C++ saved to: EBI-IntAct.graph
--- Preview of cpp_input.graph (first 5 lines) ---
3580 19159
4471 4486
4471 5130
4471 4000
4471 2616

--- Step 3: Generating .hypergraph file for C++ ---
Hypergraph file for C++ saved to: EBI_IntAct.hypergraph
--- Preview of cpp_input.hypergraph (first 5 lines) ---
4471 4486
4471 5130
2899 294
5354 2899
1087 2888

